# Moral Machine Reanalysis: Intervention Effect

The Moral Machine experiment (Awad et al., 2018) collected large-scale judgments about unavoidable autonomous-vehicle crashes, where each scenario had two outcomes depending on whether the car **swerves** or **stays on course**. In the paper's Judge mode, users selected which outcome they found preferable.

The design randomized multiple factors simultaneously. In the Supplementary description, the authors list three randomized scenario dimensions that were always varied with character attributes: **interventionism**, **relation to the AV**, and **legality**. They also describe interventionism as randomizing which group is spared if the car does nothing (stays on course).

This notebook isolates the intervention variable only. Here, `Intervention` measures whether the morally relevant trade-off is attached to **taking action (swerving)** versus **not taking action (staying)**. With this coding:
- `Intervention = 1`: passengers die if the AV swerves.
- `Intervention = 0`: passengers die if the AV stays in lane.

So the intervention AMCE is the causal shift in choice probability when the same type of dilemma is framed with a different action/inaction structure (swerve versus stay), averaging over the other randomized attributes.


### Imports


In [15]:
import os
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

# Make local package imports work whether cwd is repo root or examples/.
sys.path.insert(0, os.path.abspath('..'))

from ppi_py import ppi_ols_ci, classical_ols_ci, ppi_ols_pointestimate
from ppi_py.ppi import _ols_get_stats
from ppi_py.datasets import load_dataset


### Load reduced dataset

This follows the pattern used in other examples: data are loaded via `load_dataset`,
which downloads `moralmachine.npz` to `./data/` from Google Drive if needed.


In [16]:
dataset_folder = './data/'
data = load_dataset(dataset_folder, 'moralmachine')

y = str(data['Yhat_name'])
df = pd.DataFrame({
    'UserID': data['UserID'].astype(str),
    'weights': data['weights'],
    'Intervention': data['Intervention'],
    'Saved': data['Saved'],
    y: data['Yhat'],
})
alpha = 0.05
df.head()


,UserID,weights,Intervention,Saved,gpt4turbo_wp_Saved
0,4510867385040810.0,4.340278,0,0,0
1,4510867385040810.0,4.340278,1,1,1
2,4073176426704480.0,6.510417,0,1,1
3,4073176426704480.0,6.510417,1,0,0
4,8930084739195530.0,9.765625,0,0,0


### Intervention AMCE functions


In [17]:
def _power_analysis_stats(grads, grads_hat, inv_hessian):
    grads_ = grads - grads.mean(axis=0)
    grads_hat_ = grads_hat - grads_hat.mean(axis=0)
    cov = inv_hessian @ (grads_[:, None, :] * grads_hat_[:, :, None]).mean(axis=0) @ inv_hessian
    var = inv_hessian @ (grads_[:, None, :] * grads_[:, :, None]).mean(axis=0) @ inv_hessian
    var_hat = inv_hessian @ (grads_hat_[:, None, :] * grads_hat_[:, :, None]).mean(axis=0) @ inv_hessian
    rhos_sq = np.diag(cov) ** 2 / (np.diag(var) * np.diag(var_hat))
    return rhos_sq


def compute_amce_intervention(data, y, alpha=0.05):
    dd = data.dropna(subset=[y]).copy()
    x = dd['Intervention'].rename('Intervention')
    X = sm.add_constant(x)
    fit = sm.WLS(dd[y], X, weights=dd['weights']).fit(cov_type='cluster', cov_kwds={'groups': dd['UserID']})
    ci = fit.conf_int(alpha=alpha).loc['Intervention']
    return pd.DataFrame({
        'x': ['Intervention'],
        'y': [y],
        'beta': [fit.params['Intervention']],
        'se': [fit.bse['Intervention']],
        'lower': [ci[0]],
        'upper': [ci[1]],
    })


def compute_amce_ppi_intervention(n_data, N_data, y, alpha=0.05, lam=1.0):
    n_dd = n_data.dropna(subset=[y]).copy()
    N_dd = N_data.dropna(subset=[y]).copy()

    n_X = np.column_stack((np.ones(len(n_dd)), n_dd['Intervention'].to_numpy()))
    N_X = np.column_stack((np.ones(len(N_dd)), N_dd['Intervention'].to_numpy()))

    n_Y = n_dd['Saved'].to_numpy()
    n_Yhat = n_dd[y].to_numpy()
    n_w = n_dd['weights'].to_numpy()
    N_Yhat = N_dd[y].to_numpy()
    N_w = N_dd['weights'].to_numpy()

    beta_ppi = ppi_ols_pointestimate(X=n_X, Y=n_Y, Yhat=n_Yhat, X_unlabeled=N_X, Yhat_unlabeled=N_Yhat, w=n_w, w_unlabeled=N_w, coord=1, lam=lam)
    beta_hum = ppi_ols_pointestimate(X=n_X, Y=n_Y, Yhat=n_Yhat, X_unlabeled=N_X, Yhat_unlabeled=N_Yhat, w=n_w, w_unlabeled=N_w, lam=0)
    beta_sil = ppi_ols_pointestimate(X=N_X, Y=N_Yhat, Yhat=N_Yhat, X_unlabeled=N_X, Yhat_unlabeled=N_Yhat, w=N_w, w_unlabeled=N_w, lam=0)

    lower_CI_ppi, upper_CI_ppi = ppi_ols_ci(X=n_X, Y=n_Y, Yhat=n_Yhat, X_unlabeled=N_X, Yhat_unlabeled=N_Yhat, w=n_w, w_unlabeled=N_w, alpha=alpha, coord=1, lam=lam)
    lower_CI_hum, upper_CI_hum = classical_ols_ci(X=n_X, Y=n_Y, w=n_w, alpha=alpha)
    lower_CI_sil, upper_CI_sil = classical_ols_ci(X=N_X, Y=N_Yhat, w=N_w, alpha=alpha)

    z = stats.norm.ppf(1 - alpha / 2)
    se_ppi = (upper_CI_ppi[1] - lower_CI_ppi[1]) / (2 * z)
    se_hum = (upper_CI_hum[1] - lower_CI_hum[1]) / (2 * z)
    se_sil = (upper_CI_sil[1] - lower_CI_sil[1]) / (2 * z)

    beta = sm.WLS(n_Y, n_X, weights=n_w).fit().params
    grads, grads_hat, _, inv_hessian = _ols_get_stats(pointest=beta, X=n_X, Y=n_Y, Yhat=n_Yhat, X_unlabeled=N_X, Yhat_unlabeled=N_Yhat, w=n_w, w_unlabeled=N_w, use_unlabeled=False)
    rho_sq = _power_analysis_stats(grads, grads_hat, inv_hessian)

    return pd.DataFrame({
        'x': ['Intervention'],
        'y': [y],
        'beta_ppi': [beta_ppi[1]],
        'beta_hum': [beta_hum[1]],
        'beta_sil': [beta_sil[1]],
        'se_ppi': [se_ppi],
        'se_hum': [se_hum],
        'se_sil': [se_sil],
        'lower_ppi': [lower_CI_ppi[1]],
        'upper_ppi': [upper_CI_ppi[1]],
        'lower_hum': [lower_CI_hum[1]],
        'upper_hum': [upper_CI_hum[1]],
        'lower_sil': [lower_CI_sil[1]],
        'upper_sil': [upper_CI_sil[1]],
        'ppi_corr': [np.sqrt(rho_sq[1])],
    })


### Human-only benchmark (Intervention)


In [18]:
benchmark = compute_amce_intervention(df.copy(), y, alpha=alpha)
benchmark


,x,y,beta,se,lower,upper
0,Intervention,gpt4turbo_wp_Saved,0.08542,0.001369,0.082736,0.088104


### Compact mixed-vs-human simulation (Intervention only)


In [19]:
n_human = 600
N_multipliers = [0.5, 1.0, 2.0]
reps = 8
rng = np.random.default_rng(2026)

rows = []
truth = benchmark['beta'].iloc[0]

for rep in range(reps):
    s = int(rng.integers(0, 2**31 - 1))
    human_df = df.sample(n=n_human, random_state=s)
    rest = df.drop(human_df.index)

    hum = compute_amce_intervention(human_df.copy(), y, alpha=alpha)

    for k in N_multipliers:
        N = int(n_human * k)
        silicon_df = rest.sample(n=N, random_state=s + N + 1)
        ppi = compute_amce_ppi_intervention(human_df.copy(), silicon_df.copy(), y, alpha=alpha, lam=1.0)
        rows.append({
            'rep': rep,
            'N_over_n': k,
            'n': n_human,
            'N': N,
            'beta_truth': truth,
            'beta_human': hum['beta'].iloc[0],
            'se_human': hum['se'].iloc[0],
            'beta_ppi': ppi['beta_ppi'].iloc[0],
            'se_ppi': ppi['se_ppi'].iloc[0],
        })

sim = pd.DataFrame(rows)
sim['bias_human'] = sim['beta_human'] - sim['beta_truth']
sim['bias_ppi'] = sim['beta_ppi'] - sim['beta_truth']
sim.head()


,rep,N_over_n,n,N,beta_truth,beta_human,se_human,beta_ppi,se_ppi,bias_human,bias_ppi
0,0,0.5,600,300,0.08542,0.086097,0.044414,0.001825,0.087848,0.000677,-0.083595
1,0,1.0,600,600,0.08542,0.086097,0.044414,0.149938,0.074286,0.000677,0.064518
2,0,2.0,600,1200,0.08542,0.086097,0.044414,0.199301,0.065233,0.000677,0.113881
3,1,0.5,600,300,0.08542,0.088272,0.047749,0.126078,0.085193,0.002853,0.040658
4,1,1.0,600,600,0.08542,0.088272,0.047749,0.093327,0.071165,0.002853,0.007907


### Bias and precision summary


In [20]:
summary = sim.groupby(['N_over_n'], as_index=False).agg(
    mean_bias_human=('bias_human', 'mean'),
    mean_bias_ppi=('bias_ppi', 'mean'),
    sd_beta_human=('beta_human', 'std'),
    sd_beta_ppi=('beta_ppi', 'std'),
    mean_se_human=('se_human', 'mean'),
    mean_se_ppi=('se_ppi', 'mean'),
)
summary


,N_over_n,mean_bias_human,mean_bias_ppi,sd_beta_human,sd_beta_ppi,mean_se_human,mean_se_ppi
0,0.5,-0.009448,-0.022560,0.02713,0.112761,0.046357,0.087112
1,1.0,-0.009448,0.039354,0.02713,0.034690,0.046357,0.073216
2,2.0,-0.009448,0.028381,0.02713,0.048839,0.046357,0.065305
